# 选做实验：Model Evaluation 与 Model Selection

在实际应用 Machine Learning 时，经常需要量化学习算法的表现并比较不同模型。本实验将练习课程中介绍的方法，具体包括：

- 将数据集划分为 Training Set、Cross-validation Set 和 Test Set。
- 评估 Regression 与 Classification 模型。
- 添加 Polynomial Features，改善 Linear Regression 模型的表现。
- 比较多种 Neural Network 架构。

本实验也会帮助你熟悉本周编程作业中的代码。开始吧！


## 导入软件包与设置实验环境

首先导入本实验所需的软件包。代码还包含一些设置，用于减少冗余输出并屏蔽非关键警告，使后续结果更易阅读。


In [ ]:
# for array computations and loading data
import numpy as np

# for building linear regression models and preparing data
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# for building and training neural networks
import tensorflow as tf

# custom functions
import utils

# reduce display precision on numpy arrays
np.set_printoptions(precision=2)

# suppress warnings
tf.get_logger().setLevel('ERROR')
tf.autograph.set_verbosity(0)

## Regression

首先为一个 Regression 问题开发模型。下面的数据集包含 50 个样本，每个样本有一个输入特征 `x` 和对应的目标 `y`。


In [ ]:
# Load the dataset from the text file
data = np.loadtxt('./data/data_w3_ex1.csv', delimiter=',')

# Split the inputs and outputs into separate arrays
x = data[:,0]
y = data[:,1]

# Convert 1-D arrays into 2-D because the commands later will require it
x = np.expand_dims(x, axis=1)
y = np.expand_dims(y, axis=1)

print(f"the shape of the inputs x is: {x.shape}")
print(f"the shape of the targets y is: {y.shape}")

绘制数据集，观察目标随输入变化的规律。如需查看绘图代码，可打开本 Notebook 所在目录中的 `utils.py`，查找 `plot_dataset()` 函数。


In [ ]:
# Plot the entire dataset
utils.plot_dataset(x=x, y=y, title="input vs. target")

## 划分 Training Set、Cross-validation Set 与 Test Set

此前的实验可能使用了全部数据来训练模型。在实际应用中，应留出部分数据来评估模型对新样本的 Generalization（泛化能力），以判断模型是否对 Training Set 发生了 Overfitting。

课程中介绍了常见的三部分划分方式：

- **Training Set（训练集）：** 用于训练模型。
- **Cross-validation Set（交叉验证集，也称 Validation Set、Development Set 或 Dev Set）：** 用于评估候选模型配置，例如决定添加哪些 Polynomial Features。
- **Test Set（测试集）：** 用于公正地估计最终所选模型在新样本上的表现。开发模型时不应利用 Test Set 作决策。

Scikit-learn 提供了 [`train_test_split`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)。下面将全部数据按 60%、20%、20% 的比例划分为 Training Set、Cross-validation Set 和 Test Set。


In [ ]:
# Get 60% of the dataset as the training set. Put the remaining 40% in temporary variables: x_ and y_.
x_train, x_, y_train, y_ = train_test_split(x, y, test_size=0.40, random_state=1)

# Split the 40% subset above into two: one half for cross validation and the other for the test set
x_cv, x_test, y_cv, y_test = train_test_split(x_, y_, test_size=0.50, random_state=1)

# Delete temporary variables
del x_, y_

print(f"the shape of the training set (input) is: {x_train.shape}")
print(f"the shape of the training set (target) is: {y_train.shape}\n")
print(f"the shape of the cross validation set (input) is: {x_cv.shape}")
print(f"the shape of the cross validation set (target) is: {y_cv.shape}\n")
print(f"the shape of the test set (input) is: {x_test.shape}")
print(f"the shape of the test set (target) is: {y_test.shape}")

再次绘制数据集，查看哪些点被分到了 Training Set、Cross-validation Set 或 Test Set。


In [ ]:
utils.plot_train_cv_test(x_train, y_train, x_cv, y_cv, x_test, y_test, title="input vs. target")

## 拟合 Linear Model

完成数据划分后，可以先尝试拟合一个 Linear Model。下面将逐步完成这一过程。


### Feature Scaling

本专项课程的上一门课介绍过，Feature Scaling 通常能帮助模型更快收敛，尤其是在输入特征的数值范围差异较大时。本实验稍后将添加 Polynomial Terms，届时特征范围会明显不同。例如，$x$ 约在 1,600 到 3,600 之间，而 $x^2$ 则在 256 万到 1,296 万之间。

第一个模型只使用 $x$，但可以先练习 Feature Scaling，为后续做好准备。这里使用 Scikit-learn 的 [`StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html)，计算输入的 Z-score：

$$z=\frac{x-\mu}{\sigma}.$$

其中，$\mu$ 是特征值的均值，$\sigma$ 是标准差。下面用它处理 Training Set，并再次绘图。数据的整体规律应保持不变，而 `x` 的数值范围会缩小。


In [ ]:
# Initialize the class
scaler_linear = StandardScaler()

# Compute the mean and standard deviation of the training set then transform it
X_train_scaled = scaler_linear.fit_transform(x_train)

print(f"Computed mean of the training set: {scaler_linear.mean_.squeeze():.2f}")
print(f"Computed standard deviation of the training set: {scaler_linear.scale_.squeeze():.2f}")

# Plot the results
utils.plot_dataset(x=X_train_scaled, y=y_train, title="scaled input vs. target")

### 训练 Regression 模型

接下来创建并训练 Regression 模型。本实验使用 [`LinearRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html) 类；你也可以尝试其他 [Linear Regressors](https://scikit-learn.org/stable/modules/classes.html#classical-linear-regressors)。


In [ ]:
# Initialize the class
linear_model = LinearRegression()

# Train the model
linear_model.fit(X_train_scaled, y_train )

### 评估 Regression 模型

通过计算 Training Set 和 Cross-validation Set 的误差来评估模型。按照本课程的约定，训练误差采用带 $1/2$ 系数的 Mean Squared Error（MSE）：

$$J_{train}(\vec{w},b)=\frac{1}{2m_{train}}\sum_{i=1}^{m_{train}}\left(f_{\vec{w},b}(\vec{x}_{train}^{(i)})-y_{train}^{(i)}\right)^2.$$

Scikit-learn 提供了 [`mean_squared_error()`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_squared_error.html)。根据其[文档](https://scikit-learn.org/stable/modules/model_evaluation.html#mean-squared-error)，该函数除以样本数 `m`，而不是 `2*m`。Course 1 的 Cost Function 采用除以 `2m` 的约定，因此下面将函数结果再除以 2，以匹配上述公式。这个常数因子不影响最优参数。代码还提供了 `for` 循环实现，方便核对结果。

另请注意：模型是在经过 Z-score 标准化的数据上训练的，因此评估时也应输入缩放后的 Training Set，而非原始值。


In [ ]:
# Feed the scaled training set and get the predictions
yhat = linear_model.predict(X_train_scaled)

# Use scikit-learn's utility function and divide by 2
print(f"training MSE (using sklearn function): {mean_squared_error(y_train, yhat) / 2}")

# for-loop implementation
total_squared_error = 0

for i in range(len(yhat)):
    squared_error_i  = (yhat[i] - y_train[i])**2
    total_squared_error += squared_error_i                                              

mse = total_squared_error / (2*len(yhat))

print(f"training MSE (for-loop implementation): {mse.squeeze()}")

Cross-validation Set 的误差采用基本相同的公式：

$$J_{cv}(\vec{w},b)=\frac{1}{2m_{cv}}\sum_{i=1}^{m_{cv}}\left(f_{\vec{w},b}(\vec{x}_{cv}^{(i)})-y_{cv}^{(i)}\right)^2.$$

Cross-validation Set 也需要缩放。**必须使用 Training Set 的均值和标准差**计算其 Z-score，才能保证变换方式与模型训练时一致。考虑以下例子：

- Training Set 中某个特征值为 `500`，标准化后是 `0.5`。
- 训练后，模型能将输入 `x=0.5` 准确映射到目标 `y=300`。
- 部署后，用户输入一个原始特征值同样为 `500` 的样本。
- 如果改用另一组均值和标准差，它可能不再被变换为 `0.5`，模型也就可能给出错误预测。

下面复用之前的 `StandardScaler`，仅调用 [`transform()`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html#sklearn.preprocessing.StandardScaler.transform)，而不调用会重新估计统计量的 [`fit_transform()`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html#sklearn.preprocessing.StandardScaler.fit_transform)。


In [ ]:
# Scale the cross validation set using the mean and standard deviation of the training set
X_cv_scaled = scaler_linear.transform(x_cv)

print(f"Mean used to scale the CV set: {scaler_linear.mean_.squeeze():.2f}")
print(f"Standard deviation used to scale the CV set: {scaler_linear.scale_.squeeze():.2f}")

# Feed the scaled cross validation set
yhat = linear_model.predict(X_cv_scaled)

# Use scikit-learn's utility function and divide by 2
print(f"Cross validation MSE: {mean_squared_error(y_cv, yhat) / 2}")

## 添加 Polynomial Features

从之前的图中可以看到：`x` 较小时，目标 `y` 上升更快；随着 `x` 增大，曲线逐渐变平。因此，直线可能不是最佳选择。已经得到 Linear Model 的训练与 Cross-validation MSE 后，可以尝试添加 Polynomial Features，看看表现是否改善。代码整体相似，只需增加几个预处理步骤。


### 创建额外特征

首先根据 Training Set 生成 Polynomial Features。下面使用 [`PolynomialFeatures`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html) 创建输入 `x` 的平方特征，即设置 `degree=2`。


In [ ]:
# Instantiate the class to make polynomial features
poly = PolynomialFeatures(degree=2, include_bias=False)

# Compute the number of features and transform the training set
X_train_mapped = poly.fit_transform(x_train)

# Preview the first 5 elements of the new training set. Left column is `x` and right column is `x^2`
# Note: The `e+<number>` in the output denotes how many places the decimal point should 
# be moved. For example, `3.24e+03` is equal to `3240`
print(X_train_mapped[:5])

然后像之前一样缩放输入，缩小数值范围。


In [ ]:
# Instantiate the class
scaler_poly = StandardScaler()

# Compute the mean and standard deviation of the training set then transform it
X_train_mapped_scaled = scaler_poly.fit_transform(X_train_mapped)

# Preview the first 5 elements of the scaled training set.
print(X_train_mapped_scaled[:5])

接着训练模型，并使用 Cross-validation Set 评估表现。对 Cross-validation Set 必须应用与 Training Set 相同的变换：添加相同次数的 Polynomial Features，再使用训练时确定的统计量缩放。


In [ ]:
# Initialize the class
model = LinearRegression()

# Train the model
model.fit(X_train_mapped_scaled, y_train )

# Compute the training MSE
yhat = model.predict(X_train_mapped_scaled)
print(f"Training MSE: {mean_squared_error(y_train, yhat) / 2}")

# Add the polynomial features to the cross validation set
X_cv_mapped = poly.transform(x_cv)

# Scale the cross validation set using the mean and standard deviation of the training set
X_cv_mapped_scaled = scaler_poly.transform(X_cv_mapped)

# Compute the cross validation MSE
yhat = model.predict(X_cv_mapped_scaled)
print(f"Cross validation MSE: {mean_squared_error(y_cv, yhat) / 2}")

添加二次项后，Training Set 和 Cross-validation Set 的 MSE 都明显改善。可以继续添加更高次项，比较哪个模型表现最好。如课程中所示，可以构建以下十个模型：

<img src="images/C2_W3_poly.png" style="width:50%;" alt="不同次数的 Polynomial Regression 模型">

将前面各步骤放入循环，即可比较多个配置。下面的实现最高使用 `degree=10`，最后绘图比较各模型的结果。


In [ ]:
# Initialize lists to save the errors, models, and feature transforms
train_mses = []
cv_mses = []
models = []
polys = []
scalers = []

# Loop over 10 times. Each adding one more degree of polynomial higher than the last.
for degree in range(1,11):
    
    # Add polynomial features to the training set
    poly = PolynomialFeatures(degree, include_bias=False)
    X_train_mapped = poly.fit_transform(x_train)
    polys.append(poly)
    
    # Scale the training set
    scaler_poly = StandardScaler()
    X_train_mapped_scaled = scaler_poly.fit_transform(X_train_mapped)
    scalers.append(scaler_poly)
    
    # Create and train the model
    model = LinearRegression()
    model.fit(X_train_mapped_scaled, y_train )
    models.append(model)
    
    # Compute the training MSE
    yhat = model.predict(X_train_mapped_scaled)
    train_mse = mean_squared_error(y_train, yhat) / 2
    train_mses.append(train_mse)
    
    # Add polynomial features and scale the cross validation set
    X_cv_mapped = poly.transform(x_cv)
    X_cv_mapped_scaled = scaler_poly.transform(X_cv_mapped)
    
    # Compute the cross validation MSE
    yhat = model.predict(X_cv_mapped_scaled)
    cv_mse = mean_squared_error(y_cv, yhat) / 2
    cv_mses.append(cv_mse)
    
# Plot the results
degrees=range(1,11)
utils.plot_train_cv_mses(degrees, train_mses, cv_mses, title="degree of polynomial vs. train and CV MSEs")

### 选择最佳 Polynomial Regression 模型

理想模型在 Training Set 和 Cross-validation Set 上都表现良好，说明它学到了训练数据的规律，同时没有严重 Overfitting。使用本实验的默认设置时，`degree=1` 到 `degree=2` 之间的 Cross-validation Error 会显著下降，随后直到 `degree=5` 都相对平稳。继续增加次数后，Cross-validation Error 总体上开始恶化。因此，可以选择 `cv_mse` 最低的模型。


In [ ]:
# Get the model with the lowest CV MSE (add 1 because list indices start at 0)
# This also corresponds to the degree of the polynomial added
degree = np.argmin(cv_mses) + 1
print(f"Lowest CV MSE is found in the model with degree={degree}")

最后计算 Test Set 的 MSE，作为可报告的 Generalization Error。仍需采用与 Training Set 和 Cross-validation Set 相同的数据变换。


In [ ]:
# Add polynomial features to the test set
X_test_mapped = polys[degree-1].transform(x_test)

# Scale the test set
X_test_mapped_scaled = scalers[degree-1].transform(X_test_mapped)

# Compute the test MSE
yhat = models[degree-1].predict(X_test_mapped_scaled)
test_mse = mean_squared_error(y_test, yhat) / 2

print(f"Training MSE: {train_mses[degree-1]:.2f}")
print(f"Cross Validation MSE: {cv_mses[degree-1]:.2f}")
print(f"Test MSE: {test_mse:.2f}")

## Neural Networks

选择 Neural Network 架构时，也可以采用相同的 Model Selection 流程。本节将构建下图中的模型，处理上述同一个 Regression 任务。

<img src="images/C2_W3_NN_Arch.png" style="width:40%;" alt="三种候选 Neural Network 架构">


### 准备 Neural Network 的数据

继续使用上一节划分好的 Training Set、Cross-validation Set 和 Test Set。Neural Network 能学习非线性关系，因此可以不添加 Polynomial Features。下面仍保留了相关代码，方便你之后尝试并比较效果。默认 `degree=1`，表示直接使用 `x_train`、`x_cv` 和 `x_test`，不添加额外的高次项。


In [ ]:
# Add polynomial features
degree = 1
poly = PolynomialFeatures(degree, include_bias=False)
X_train_mapped = poly.fit_transform(x_train)
X_cv_mapped = poly.transform(x_cv)
X_test_mapped = poly.transform(x_test)

接下来缩放输入特征，帮助 Gradient Descent 更快收敛。注意，Cross-validation Set 和 Test Set 只调用 `transform()`，不调用 `fit_transform()`，因此仍使用 Training Set 的均值和标准差。


In [ ]:
# Scale the features using the z-score
scaler = StandardScaler()
X_train_mapped_scaled = scaler.fit_transform(X_train_mapped)
X_cv_mapped_scaled = scaler.transform(X_cv_mapped)
X_test_mapped_scaled = scaler.transform(X_test_mapped)

### 构建并训练 Neural Networks

接下来创建前面展示的 Neural Network 架构。实现位于 `utils.py` 的 `build_models()` 函数中，可按需查看或修改。下面在循环中调用它并训练模型，同时记录各模型的训练误差与 Cross-validation Error。


In [ ]:
# Initialize lists that will contain the errors for each model
nn_train_mses = []
nn_cv_mses = []

# Build the models
nn_models = utils.build_models()

# Loop over the the models
for model in nn_models:
    
    # Setup the loss and optimizer
    model.compile(
    loss='mse',
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.1),
    )

    print(f"Training {model.name}...")
    
    # Train the model
    model.fit(
        X_train_mapped_scaled, y_train,
        epochs=300,
        verbose=0
    )
    
    print("Done!\n")

    
    # Record the training MSEs
    yhat = model.predict(X_train_mapped_scaled)
    train_mse = mean_squared_error(y_train, yhat) / 2
    nn_train_mses.append(train_mse)
    
    # Record the cross validation MSEs 
    yhat = model.predict(X_cv_mapped_scaled)
    cv_mse = mean_squared_error(y_cv, yhat) / 2
    nn_cv_mses.append(cv_mse)

    
# print results
print("RESULTS:")
for model_num in range(len(nn_train_mses)):
    print(
        f"Model {model_num+1}: Training MSE: {nn_train_mses[model_num]:.2f}, " +
        f"CV MSE: {nn_cv_mses[model_num]:.2f}"
        )

根据记录的误差选择适合当前应用的模型。查看上面的结果，判断是否同意下方 `model_num` 的选择。最后计算 Test Error，估计模型对新样本的 Generalization 能力。


In [ ]:
# Select the model with the lowest CV MSE
model_num = 3

# Compute the test MSE
yhat = nn_models[model_num-1].predict(X_test_mapped_scaled)
test_mse = mean_squared_error(y_test, yhat) / 2

print(f"Selected Model: {model_num}")
print(f"Training MSE: {nn_train_mses[model_num-1]:.2f}")
print(f"Cross Validation MSE: {nn_cv_mses[model_num-1]:.2f}")
print(f"Test MSE: {test_mse:.2f}")

## Classification

本实验最后一部分将在 Classification 任务上练习 Model Evaluation 与 Model Selection。流程基本相同，主要区别在于误差的计算方式。


### 加载 Classification 数据集

首先加载一个 Binary Classification 数据集。它包含 200 个样本，每个样本有两个输入特征 `x1`、`x2`，目标 `y` 为 `0` 或 `1`。


In [ ]:
# Load the dataset from a text file
data = np.loadtxt('./data/data_w3_ex2.csv', delimiter=',')

# Split the inputs and outputs into separate arrays
x_bc = data[:,:-1]
y_bc = data[:,-1]

# Convert y into 2-D because the commands later will require it (x is already 2-D)
y_bc = np.expand_dims(y_bc, axis=1)

print(f"the shape of the inputs x is: {x_bc.shape}")
print(f"the shape of the targets y is: {y_bc.shape}")

绘制数据集，观察两个类别的样本如何分布。


In [ ]:
utils.plot_bc_dataset(x=x_bc, y=y_bc, title="x1 vs. x2")

### 划分并准备 Classification 数据集

继续按 60%、20%、20% 的比例划分 Training Set、Cross-validation Set 和 Test Set，并像上一节一样进行 Feature Scaling。


In [ ]:
from sklearn.model_selection import train_test_split

# Get 60% of the dataset as the training set. Put the remaining 40% in temporary variables.
x_bc_train, x_, y_bc_train, y_ = train_test_split(x_bc, y_bc, test_size=0.40, random_state=1)

# Split the 40% subset above into two: one half for cross validation and the other for the test set
x_bc_cv, x_bc_test, y_bc_cv, y_bc_test = train_test_split(x_, y_, test_size=0.50, random_state=1)

# Delete temporary variables
del x_, y_

print(f"the shape of the training set (input) is: {x_bc_train.shape}")
print(f"the shape of the training set (target) is: {y_bc_train.shape}\n")
print(f"the shape of the cross validation set (input) is: {x_bc_cv.shape}")
print(f"the shape of the cross validation set (target) is: {y_bc_cv.shape}\n")
print(f"the shape of the test set (input) is: {x_bc_test.shape}")
print(f"the shape of the test set (target) is: {y_bc_test.shape}")

In [ ]:
# Scale the features

# Initialize the class
scaler_linear = StandardScaler()

# Compute the mean and standard deviation of the training set then transform it
x_bc_train_scaled = scaler_linear.fit_transform(x_bc_train)
x_bc_cv_scaled = scaler_linear.transform(x_bc_cv)
x_bc_test_scaled = scaler_linear.transform(x_bc_test)

### 计算 Classification 模型的误差

前面的 Regression 模型使用 MSE 衡量表现。对于 Classification，可以用分类错误的样本占比作为指标。例如，5 个样本中预测错了 2 个，则错误率为 `40%`，即 `0.4`。下面分别使用 `for` 循环和 NumPy 的 [`mean()`](https://numpy.org/doc/stable/reference/generated/numpy.mean.html) 实现。


In [ ]:
# Sample model output
probabilities = np.array([0.2, 0.6, 0.7, 0.3, 0.8])

# Apply a threshold to the model output. If greater than 0.5, set to 1. Else 0.
predictions = np.where(probabilities >= 0.5, 1, 0)

# Ground truth labels
ground_truth = np.array([1, 1, 1, 1, 1])

# Initialize counter for misclassified data
misclassified = 0

# Get number of predictions
num_predictions = len(predictions)

# Loop over each prediction
for i in range(num_predictions):
    
    # Check if it matches the ground truth
    if predictions[i] != ground_truth[i]:
        
        # Add one to the counter if the prediction is wrong
        misclassified += 1

# Compute the fraction of the data that the model misclassified
fraction_error = misclassified/num_predictions

print(f"probabilities: {probabilities}")
print(f"predictions with threshold=0.5: {predictions}")
print(f"targets: {ground_truth}")
print(f"fraction of misclassified data (for-loop): {fraction_error}")
print(f"fraction of misclassified data (with np.mean()): {np.mean(predictions != ground_truth)}")

### 构建并训练 Classification 模型

继续使用上一节的 Neural Network 架构，重新调用 `build_models()` 创建新的模型实例。

按照上周推荐的方式，Output Layer 使用 `linear` Activation，并在 Loss Function 中设置 `from_logits=True`。这是 Binary Classification 问题，因此使用 [Binary Cross-entropy Loss](https://www.tensorflow.org/api_docs/python/tf/keras/losses/BinaryCrossentropy)。

训练完成后，用 [Sigmoid Function](https://www.tensorflow.org/api_docs/python/tf/math/sigmoid) 将模型输出转换为概率，再设置阈值，计算 Training Set 和 Cross-validation Set 中分类错误的样本占比。下面的代码包含上述全部步骤。


In [ ]:
# Initialize lists that will contain the errors for each model
nn_train_error = []
nn_cv_error = []

# Build the models
models_bc = utils.build_models()

# Loop over each model
for model in models_bc:
    
    # Setup the loss and optimizer
    model.compile(
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    )

    print(f"Training {model.name}...")

    # Train the model
    model.fit(
        x_bc_train_scaled, y_bc_train,
        epochs=200,
        verbose=0
    )
    
    print("Done!\n")
    
    # Set the threshold for classification
    threshold = 0.5
    
    # Record the fraction of misclassified examples for the training set
    yhat = model.predict(x_bc_train_scaled)
    yhat = tf.math.sigmoid(yhat)
    yhat = np.where(yhat >= threshold, 1, 0)
    train_error = np.mean(yhat != y_bc_train)
    nn_train_error.append(train_error)

    # Record the fraction of misclassified examples for the cross validation set
    yhat = model.predict(x_bc_cv_scaled)
    yhat = tf.math.sigmoid(yhat)
    yhat = np.where(yhat >= threshold, 1, 0)
    cv_error = np.mean(yhat != y_bc_cv)
    nn_cv_error.append(cv_error)

# Print the result
for model_num in range(len(nn_train_error)):
    print(
        f"Model {model_num+1}: Training Set Classification Error: {nn_train_error[model_num]:.5f}, " +
        f"CV Set Classification Error: {nn_cv_error[model_num]:.5f}"
        )

根据上面的输出选择表现最好的模型。如果 Cross-validation Error 相同，可以增加一个选择标准，例如选择训练误差更低的模型。更常见的做法是选择较小的模型，以节省计算资源。本例中 Model 1 最小，Model 3 最大。

最后计算 Test Error，报告模型的 Generalization Error。


In [ ]:
# Select the model with the lowest error
model_num = 3

# Compute the test error
yhat = models_bc[model_num-1].predict(x_bc_test_scaled)
yhat = tf.math.sigmoid(yhat)
yhat = np.where(yhat >= threshold, 1, 0)
nn_test_error = np.mean(yhat != y_bc_test)

print(f"Selected Model: {model_num}")
print(f"Training Set Classification Error: {nn_train_error[model_num-1]:.4f}")
print(f"CV Set Classification Error: {nn_cv_error[model_num-1]:.4f}")
print(f"Test Set Classification Error: {nn_test_error:.4f}")

## 实验回顾

本实验练习了评估模型表现、比较并选择不同模型配置的方法。你将数据划分为 Training Set、Cross-validation Set 和 Test Set，并了解了它们各自的用途。后续课程将通过诊断 Bias 与 Variance，介绍更多改善模型的方法。
